# L10c: Stochastic Multi-Armed Bandit Problems
In this lecture, we continue our exploration of Reinforcement Learning (RL), a branch of machine learning focused on how agents should take actions in an environment to maximize cumulative reward over time. We focus on multi-armed bandit problems, where the agent learns only from the reward of each action it takes.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Balance exploration and exploitation:__ An agent that learns from rewards must choose between trying actions to learn more about them (exploration) and taking the action that looks best so far (exploitation). We'll see why an agent needs both, and how each algorithm in this lecture balances them.
> * __Solve multi-armed bandit problems:__ In a multi-armed bandit problem, an agent repeatedly chooses one of several actions (arms), each with an unknown reward distribution, and tries to collect as much reward as possible. We'll work through the explore-first, epsilon-greedy, UCB1, and Thompson sampling algorithms for choosing the next arm.
> * __Compare bandit algorithms with regret:__ Regret is the reward an agent loses by not always choosing the best arm. We'll define the regret of a bandit algorithm and compare the bounds on how fast the regret of each algorithm can grow with the number of rounds.

At its core, RL is about learning from doing: an agent observes the state of its environment, takes actions, i.e., makes decisions, receives rewards, and updates its knowledge to improve future decision-making. Let’s get started!
___

## Exploration vs. Exploitation
In reinforcement learning, the problem on the surface is deceptively simple: an agent is in a state $s$ and can take an action $a\in A_{s}$, where $A_{s}$ is the set of actions currently available to the agent. The agent chooses an action $a$, implements it, receives a reward $r$, and transitions to a new state $s^{\prime}$. The goal is to learn a policy that maximizes cumulative reward over time, i.e., the best possible action in each state.

But when you think about it, the problem is actually quite complex. The agent must make decisions based on incomplete information and balance two competing objectives: exploration and exploitation. The exploration vs. exploitation trade-off is a fundamental challenge that agents must navigate. It involves balancing two competing objectives:
1. **Exploration**: Trying new actions to discover potentially better rewards. This is essential for learning about the environment and finding optimal policies. Taking purely random actions, or actions that have not been tried often, to gather information about their outcomes and rewards is an example of exploration.
2. **Exploitation**: Leveraging current knowledge to maximize immediate payoff. This involves choosing actions that have previously yielded high rewards based on the agent's experience. However, if the agent only exploits, it may miss out on discovering better actions that could yield higher rewards in the long run. The agent never tries anything new; how boring!

Striking the right balance between exploration and exploitation is crucial for learning an optimal policy that performs well both now and in the long run. If an agent explores too much, it may miss out on immediate rewards. If it exploits too much, it may fail to discover better long-term strategies.

The exploration-exploitation trade-off is often formalized in algorithms that guide the agent's decision-making process. These algorithms provide principled strategies for managing the trade-off, ensuring that the agent can learn effectively while maximizing cumulative reward over time.
___

## What Is a Bandit Problem?  
A bandit problem is a class of online (sequential) decision-making tasks in which an agent repeatedly chooses among $K$ options (called _arms_) and receives a reward based on that choice.

The agent chooses from $K$ alternatives (somehow) and executes the desired action. At each round $t$, the agent pulls one arm and observes a reward $r_{t}$. Good pulls yield higher rewards; poor pulls yield lower rewards (or even losses). The agent’s goal is to maximize cumulative reward over time.

Here are a few examples of applications of bandit problems:
* __Clinical Trials__: Balances learning about new treatments (exploration) with assigning patients to the current best therapy (exploitation).
* __Financial Portfolio Design__: Dynamically allocates capital across assets to maximize returns while testing novel investments.
* __Adaptive Routing__: Chooses network paths to minimize delay, trading off probing unknown routes against using established fast ones.
* __Recommendation Systems__: Iteratively selects items to display (like movies or products), balancing novel suggestions against proven favorites.

__Additional Resources__: Our lecture notes for this week were inspired by Chapter 1 of "Introduction to Multi-Armed Bandits" by Aleksandrs Slivkins. This is an excellent resource (albeit quite technical) for learning more about bandit problems. [The book is available online](https://arxiv.org/abs/1904.07272). We also drew material from the [Bandit problem Thompson sampling tutorial by Russo et al., 2020](https://arxiv.org/abs/1707.02038).

In the stochastic multi-armed bandit problem, the agent must choose an action $a$ from the set of all possible actions $\mathcal{A}$, where $|\mathcal{A}| = K$ during each round $t = 1,2,\dots, T$ of the game or task. The agent receives a reward $r_{a}$ from the environment, where $r_{a}$ is sampled from some unknown distribution $\mathcal{D}_{a}$.

For $t = 1,2,\dots,T$:
1. _Aggregator_: The agent picks an action $a_{t} \in \mathcal{A}$. How the agent makes this choice is one of the main differences between algorithms for solving this problem. 
2. _Adversary_: The agent implements action $a_{t}$ and receives a reward $r_{t}\in\left[0,1\right]$ sampled from the (unknown) distribution $\mathcal{D}_{a}\mid a = a_{t}$.
3. Agent observes $r_{t}$ but nothing else. It cannot see the distribution $\mathcal{D}_{a}$; only the _adversary_ can see this.

The agent is interested in learning the mean of the reward distribution of each arm, $\mu(a) = \mathbb{E}\left[r_{t}\sim\mathcal{D}_{a}\right]$, by experimenting against the world (adversary). The goal of the agent is to maximize the total reward. However, the goal of the algorithm designer is to minimize the _regret_ of the algorithm that the agent uses to choose $a\in\mathcal{A}$.

### Regret
Regret measures the difference between what could have been achieved by always making the best decision, i.e., the decision that maximizes reward (in hindsight), and what the agent actually chooses to do during each round. 
> __Perspective__: Regret is a property of the algorithm; the agent itself only cares about reward. Each decision-making framework the agent employs may lead to a different bound on regret. Thus, the goal of the algorithm designer is to minimize the regret of the agent's algorithm.

__Definition__: _Regret_. Let $\mu^{\star}$ be the mean of the best arm, i.e., $\mu^{\star} = \max_{a\in\mathcal{A}}\mu(a)$ after playing the game for $T$ rounds. The regret $R(T)$ of an algorithm after $T$ rounds is defined as:
$$
\begin{align*}
R(T) = T\cdot\mu^{\star} - \sum_{t=1}^{T}r_{t}
\end{align*}
$$
The first term is the expected reward from always choosing the best arm over the $T$ rounds. The second term is the total reward obtained by the agent over the $T$ rounds, where $r_{t}$ is the reward received at round $t$ for the action chosen by the agent.

### Explore-First Exploration
A straightforward approach to the multi-armed bandit problem is to explore each arm equally. This is called _uniform exploration_ or the explore-first algorithm. In this approach, the agent begins with a purely _exploratory phase_, pulling each arm $N$ times. After this exploration phase, the agent selects the arm with the highest mean reward for the rest of the game. This is called the _exploitation phase_.

#### Explore-First Algorithm
The agent has $K$ arms, $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T$. The agent uses the following algorithm to choose which arm to pull during each round:
1. _Initialization_: For each arm $a\in\mathcal{A}$, set $N_{a} = (T/K)^{2/3}\cdot\mathcal{O}\left(\log{T}\right)^{1/3}$ (the number of times we try action $a$).
2. _Exploration_: Play each arm $a\in\mathcal{A}$ for $N_{a}$ rounds and record the rewards. After the exploration phase, select the arm $a^{\star}$ with the highest mean reward (break ties arbitrarily).
3. _Exploitation_: Play arm $a^{\star}$ for the remaining rounds.

__Theorem__: The _expected_ regret over $T$ rounds of the _uniform exploration_ algorithm is bounded by $\mathbb{E}\left[R(T)\right]\leq{T}^{2/3}\times\mathcal{O}\left(K\cdot\log{T}\right)^{1/3}$, where $K$ is the number of arms, $T$ is the total number of rounds and $N = (T/K)^{2/3}\cdot\mathcal{O}\left(\log{T}\right)^{1/3}$ is the number of rounds in the exploration phase for each action (choice).

### Epsilon-Greedy Exploration
One issue with the _uniform exploration_ algorithm is that it may not be the best choice for all problems. For example, performance in the exploration phase may be _bad_ if many of the arms have a large gap $\Delta({a})$:
* _What is this gap?_ Let the (true) mean reward for each arm be $\mu(a) = \mathbb{E}\left[r_{t}\sim\mathcal{D}_{a}\right]$, where $a\in\mathcal{A}$. The _best_ mean reward over the actions is $\mu^{\star} = \max_{a\in\mathcal{A}}\mu_{a}$. Then, the gap $\Delta({a}) = \mu^{\star} - \mu(a)$ is the difference between the mean reward of the best arm and the mean reward of arm $a$. If the gap is _large_, the agent may miss out on many rewards by exploring each arm equally.

With a large gap, it may be better to spread out (and interleave) the exploration and exploitation phases of the arms. This is the idea behind the _epsilon-greedy_ algorithm. In this algorithm, the agent chooses the best arm with probability $1-\epsilon$ and a random arm with probability $\epsilon$. This allows the agent to explore the arms more evenly and may lead to better performance in cases where the gap is large.

While [Slivkins](https://arxiv.org/abs/1904.07272) doesn't give a reference for the epsilon-greedy algorithm, other sources point to (at least in part) to [Thompson and Thompson sampling, proposed in 1933 in the context of drug trials](https://arxiv.org/abs/1707.02038).

#### Epsilon-Greedy Algorithm
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. _Initialize_: Roll a random number $u\in\left[0,1\right]$ and compute a threshold $\epsilon_{t}\sim{t}^{-1/3}$. Note that in other sources, $\epsilon$ is a constant, not a function of $t$.
2. _Exploration_: If $u\leq\epsilon_{t}$, choose a random (uniform) arm $a_{t}\in\mathcal{A}$. Execute action $a_{t}$ and receive a reward $r_{t}$ from the _adversary_ (nature). 
3. _Exploitation_: Else if $u>\epsilon_{t}$, choose action $a^{\star}$ (the action with the highest average reward so far, the greedy choice). Execute action $a^{\star}_{t}$ and receive a reward $r_{t}$ from the _adversary_ (nature).
4. Update the list of rewards for the arm chosen in this round.

__Theorem__: The epsilon-greedy algorithm with exploration probability $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$ achieves a regret bound of $\mathbb{E}\left[R(t)\right]\leq{t}^{2/3}\cdot\mathcal{O}\left(K\cdot\log(t)\right)^{1/3}$ for each round $t$.

### Optimism Under Uncertainty
Let's consider the final approach for solving bandit problems for today: the optimism under uncertainty algorithm. The key assumption of this approach:
* __Assumption__: Assume each arm is as good as it can be given the observations so far, and choose the best arm based on these optimistic estimates. This intuition leads to the `UCB1` algorithm.

Given a history of rewards and the number of pulls for each arm, the `UCB1` algorithm calculates an upper confidence bound (UCB) and uses it to decide which arm to pull. 

__Definition__: Upper Confidence Bound (UCB). During each round $t =1,2, \dots, T$ the `UCB1` algorithm maximizes the sum $\bar{\mu}(a)+U(a,t)$ where $\bar{\mu}(a)$ is the _estimated_ mean return of arm $a\in\mathcal{A}$ at time $t$ and $U(a,t)$ is the _confidence radius_ of arm $a\in\mathcal{A}$ at time $t$, so the sum is an upper confidence bound on the mean reward of arm $a$:
$$
\begin{align*}
U(a,t) = \sqrt{\frac{2\log(t)}{N_{a}}}
\end{align*}
$$
where $N_{a}$ is the number of times that arm $a\in\mathcal{A}$ has been pulled up to time $t$. The `UCB1` algorithm chooses the arm $a^{\star}$ that maximizes the sum $\bar{\mu}(a)+U(a,t)$ during each round $t$. The `UCB1` algorithm was originally proposed by Auer, Cesa-Bianchi, and Fischer in 2002.

#### UCB1 Algorithm
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and a total number of rounds is $T\gg{K}$.

_Initialization_: Pull each arm $a\in\mathcal{A}$ once and record the rewards. For each arm $a\in\mathcal{A}$, set $N_{a} = 1$ and $\bar{\mu}(a) = r_{a}$.

For rounds $t = K+1,K+2,\dots,T$:
1. Compute the upper confidence bound $U(a,t)$ for each arm $a\in\mathcal{A}$.
2. Choose the best arm $a^{\star} = \text{arg}\max\,\left\{\bar{\mu}(a)+U(a,t)\mid\,a\in\mathcal{A}\right\}$ at time $t$.
3. Execute action $a^{\star}$ and receive a reward $r_{t}$ from the _adversary_ (nature).
4. Update the estimated mean reward $\bar{\mu}(a^{\star})$ and the number of pulls $N_{a^{\star}}$.

__Theorem__: The `UCB1` algorithm achieves a regret bound for $K$ arms of $\mathbb{E}\left[R(t)\right]\leq\mathcal{O}\left(\sqrt{KT\cdot\log(T)}\right)$ 
over $T$ rounds.
___

## Binary Bernoulli Bandit Problem
The binary Bernoulli bandit problem is a special case of the stochastic bandit problem where the reward for taking action $a\in\mathcal{A}$ is binary, $r_{t} \in \left\{0,1\right\}$. The probability of getting reward `1` is unknown and needs to be estimated. The goal is to maximize the expected reward by selecting the best action at each time step.

* _Difference_: Unlike a completely general stochastic bandit problem, the binary Bernoulli bandit problem assumes the _agent models how the world responds_ using a (deceptively) simple reward distribution, [the Bernoulli distribution](https://en.wikipedia.org/wiki/Bernoulli_distribution). Thus, _the agent has a model of the world_ (which is so super cool!).
* _Binary_: The reward distribution is binary, but the experiment represented by the action $a$ can be a complex statement or function that _evaluates_ to a boolean value. Thus, we can model many complex scenarios that evaluate to `true` or `false`.

The Bernoulli distribution is a discrete probability distribution that returns a value of `1` with probability $p$ and value `0` with probability $1-p$. The probability mass function of the Bernoulli distribution is given by:
$$
\begin{equation*}
\texttt{Bern}(r; p) = \begin{cases}
p & \text{if } r = 1,\\
1-p & \text{if } r = 0.
\end{cases}
\end{equation*}
$$
where $r\in\left\{0,1\right\}$ is the reward and $p\in[0,1]$ is the probability of getting reward `r = 1`. The expected reward of $X\sim\texttt{Bern}(r;p)$ is given by: $\mathbb{E}[X] = p$ and the variance is given by: $\text{Var}[X] = p(1-p)$. 
* _Ready to get your mind blown_? Ok, so here is the _cool part_: the agent models the parameter $p$ using a _probability distribution_ (e.g., [a Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution)) and updates this distribution as it observes rewards. This is the essence of the [Bayesian approach to bandit problems](https://onlinelibrary.wiley.com/doi/10.1002/asmb.874).

The [Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) is a natural choice for this model:

> __Beta distribution:__ The distribution $\text{Beta}(\alpha,\beta)$ describes an unknown probability $p\in[0,1]$. The parameters $\alpha>0$ and $\beta>0$ set its shape, and its mean is $\alpha/(\alpha+\beta)$. Suppose our belief about $p$ before we see any rewards (the __prior__) is $\text{Beta}(\alpha,\beta)$, where $\text{Beta}(1,1)$ is the uniform distribution on $[0,1]$. After we observe $S$ successes and $F$ failures, our updated belief (the __posterior__) is again a Beta distribution, $\text{Beta}(\alpha+S,\,\beta+F)$.

Yeah. That's cool. But how do we solve this problem?

### $\epsilon$-Greedy Binary Bernoulli Bandit
The $\epsilon$-greedy algorithm is simple and effective for solving the binary Bernoulli bandit problem.

The algorithm selects the _best action_ with probability $1-\epsilon$ and selects a random action with probability $\epsilon$. The pseudocode for the $\epsilon$-greedy algorithm is given below [(a more detailed version can be found here)](https://github.com/varnerlab/CHEME-5820-Lectures-Spring-2025/blob/main/lectures/week-7/L7c/docs/BBBPcode.pdf):

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$. Initialize the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha(a) = 1$ and $\beta(a) = 1$, and set the number of successes and failures to $S(a) = F(a) = 0$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. _Initialize_: Roll a random number $u\in\left[0,1\right]$ and compute a threshold $\epsilon_{t}={t^{-1/3}}\cdot\left(K\cdot\log(t)\right)^{1/3}$.
2. _Exploration_: If $u\leq\epsilon_{t}$, choose a random (uniform) arm $a_{t}\in\mathcal{A}$. Execute the action $a_{t}$ and receive a reward $r_{t} \in \left\{0,1\right\}$ from the _adversary_ (nature).
3. _Exploitation_: Else if $u>\epsilon_{t}$, choose action $a_{t} = a^{\star}$, the action with the _highest expected probability of success_ (greedy choice), using the agent's model of the world. The highest probability action is: $a^{\star} = \arg\max_{a\in\mathcal{A}}\left\{\frac{\alpha(a) + S(a)}{\alpha(a) + \beta(a) + S(a) + F(a)}\right\}$, the posterior mean, where $S(a)$ and $F(a)$ are the number of successes and failures for arm $a$. Execute the action $a_{t}$ and receive a reward $r_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature).
4. Update the number of successes $S(a_{t})$ and failures $F(a_{t})$ for the arm $a_{t}$ chosen in step 2 or 3, using its reward $r_{t}$:
$$
\begin{equation*}
S(a_{t}) \gets S(a_{t}) + r_{t},\quad F(a_{t}) \gets F(a_{t}) + (1-r_{t})
\end{equation*}
$$

Using a model of the world allows the agent to make decisions about which actions to take. This is the essence of the Bayesian approach to bandit problems. The agent has a model of the likely reward distribution for _each_ action and uses this model to select the best action at each time step.

### Thompson Sampling Binary Bernoulli Bandit
An alternative Bayesian approach is Thompson sampling, which samples from the posterior distribution to make decisions.

> The algorithm selects the _best action_ by sampling from the posterior distribution for each arm and choosing the one with the highest sample.

#### Pseudo-code
The agent has $K$ arms (choices), $\mathcal{A} = \left\{1,2,\dots,K\right\}$, and the total number of rounds is $T\gg{K}$. Initialize the parameters of [the Beta distribution](https://en.wikipedia.org/wiki/Beta_distribution) for each arm $a\in\mathcal{A}$ to $\alpha(a) = 1$ and $\beta(a) = 1$, and set the number of successes and failures to $S(a) = F(a) = 0$. The agent uses the following algorithm to choose which arm to pull (which action to take) during each round:

For $t = 1,2,\dots,T$:
1. Sample from the posterior for each arm: $p(a)\sim\text{Beta}\left(\alpha(a)+S(a),\,\beta(a)+F(a)\right)$ for every $a\in\mathcal{A}$, where $S(a)$ and $F(a)$ are the number of successes and failures for arm $a$.
2. Choose the action with the highest sampled probability: $a^{\star}_{t} = \arg\max_{a\in\mathcal{A}}\left\{p(a)\right\}$.
3. Execute the action $a^{\star}_{t}$ and receive a reward $r^{\star}_{t}\in\left\{0,1\right\}$ from the _adversary_ (nature).
4. Update the number of successes $S(a^{\star}_{t})$ and failures $F(a^{\star}_{t})$ for the chosen arm $a^{\star}_{t}$ using the reward $r^{\star}_{t}$:
$$
\begin{equation*}
S(a^{\star}_{t}) \gets S(a^{\star}_{t}) + r^{\star}_{t},\quad F(a^{\star}_{t}) \gets F(a^{\star}_{t}) + (1-r^{\star}_{t})
\end{equation*}
$$

Thompson sampling is fully Bayesian and often performs better than $\epsilon$-greedy in practice.

___

## Lab
In lab `L10d`, we run the explore-first, epsilon-greedy, and UCB1 algorithms on a bandit with seven arms and compare their regret with the bounds above. We then implement Thompson sampling for a binary Bernoulli bandit and watch the posterior of each arm concentrate around its true success probability.

___

## Summary
In this lecture, we introduced the stochastic multi-armed bandit problem and four algorithms for balancing exploration and exploitation.

> __Key Takeaways:__
>
> * __Exploration and exploitation:__ We saw that an agent learning from rewards must both try actions to learn about them and take the action that looks best so far. Too much exploration wastes rewards on poor arms, while too little can lock the agent onto an arm that only looked good by chance.
>
> * __Bandit algorithms:__ We worked through four ways to choose the next arm: explore every arm first and then commit, explore at random with a shrinking probability (epsilon-greedy), add an optimism bonus to each arm's estimated reward (UCB1), and sample from a Beta model of each arm's success probability (Thompson sampling). For binary rewards, the Beta model turns the successes and failures of each arm into a distribution for its success probability, which the agent updates after every pull.
>
> * __Regret bounds:__ We measured each algorithm by its regret, the reward lost by not always choosing the best arm. The explore-first and epsilon-greedy bounds grow with the two-thirds power of the number of rounds, while the UCB1 bound grows only with about the square root, so UCB1 has the strongest guarantee of the three.

Next week, the agent's actions also change the state of its environment, which leads to Markov decision processes.

___